# 03 — Gold Model

Builds the star schema for Power BI Direct Lake.

```
Dimensions                Facts
──────────────            ──────────────────────────────────────
dim_student          →    fact_lead       (Q1: funnel per contact)
dim_course           →    fact_enrolment  (Q1+Q2: course enrolments)
dim_intake           →    fact_result     (Q2: academic outcomes)
dim_marketing_source
```

All tables use integer surrogate keys (`*_key`) for Power BI relationships.

## Dimensions

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE dim_student AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY student_number)  AS student_key,
        student_number,
        contact_id,
        student_code,
        marketing_source_id,
        first_name,
        last_name,
        email,
        CAST(date_of_birth AS DATE)   AS date_of_birth,
        address_suburb,
        address_state,
        lifecycle_status,
        hs_lifecycle_stage,
        CAST(hs_created_date AS DATE) AS hs_created_date,
        CAST(enrolment_date AS DATE)  AS enrolment_date
    FROM silver_student
""")
print(f"dim_student: {spark.table('dim_student').count()} rows")


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE dim_course AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY course_id)  AS course_key,
        course_id,
        course_code,
        course_name,
        course_level,
        duration_terms
    FROM silver_course
""")

spark.sql("""
    CREATE OR REPLACE TABLE dim_intake AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY intake_id)  AS intake_key,
        intake_id,
        intake_code,
        start_date,
        end_date,
        academic_year,
        term
    FROM silver_intake
""")

spark.sql("""
    CREATE OR REPLACE TABLE dim_marketing_source AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY marketing_source_id)  AS marketing_source_key,
        marketing_source_id,
        source_name,
        source_category
    FROM silver_marketing_source
""")

for tbl in ["dim_course", "dim_intake", "dim_marketing_source"]:
    print(f"{tbl}: {spark.table(tbl).count()} rows")


## Fact Tables

### fact_lead

One row per HubSpot contact. Answers Q1: how many leads converted to students, by marketing source?  
Students who enrolled (reached D365) resolve to a `student_key`; pure leads/applicants have `student_key = NULL`.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE fact_lead AS
    SELECT
        h.contact_id,
        ms.marketing_source_key,
        h.lifecycle_stage,
        CAST(h.created_date AS DATE)                              AS created_date,
        ds.student_key,
        ds.enrolment_date,
        DATEDIFF(ds.enrolment_date, CAST(h.created_date AS DATE)) AS days_to_enrolment
    FROM bronze_hubspot_contacts        h
    JOIN  dim_marketing_source          ms ON h.marketing_source_id = ms.marketing_source_id
    LEFT JOIN dim_student               ds ON h.contact_id           = ds.contact_id
""")
print(f"fact_lead: {spark.table('fact_lead').count()} rows")


### fact_enrolment

One row per D365 enrolment record. Answers Q1 (completion/withdrawal) and Q2 (course-level outcomes).  
`marketing_source_key` is inherited from the student dimension — NULL for walk-in students.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE fact_enrolment AS
    SELECT
        e.enrolment_id,
        ds.student_key,
        dc.course_key,
        di.intake_key,
        dms.marketing_source_key,
        e.enrolment_date,
        e.status,
        e.final_grade
    FROM silver_enrolment    e
    JOIN  dim_student         ds  ON e.student_number = ds.student_number
    JOIN  dim_course           dc  ON e.course_id      = dc.course_id
    JOIN  dim_intake           di  ON e.intake_id      = di.intake_id
    LEFT JOIN dim_marketing_source dms ON ds.marketing_source_id = dms.marketing_source_id
""")
print(f"fact_enrolment: {spark.table('fact_enrolment').count()} rows")


### fact_result

One row per Paradigm assessment result. Answers Q2: which marketing sources produce students who succeed academically?  
`marketing_source_key` is inherited from the student dimension — NULL for walk-in students.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE fact_result AS
    SELECT
        r.result_id,
        ds.student_key,
        dc.course_key,
        di.intake_key,
        dms.marketing_source_key,
        r.unit_id,
        r.mark,
        r.grade,
        r.result_date
    FROM silver_result          r
    JOIN  dim_student            ds  ON r.student_number  = ds.student_number
    JOIN  dim_course             dc  ON r.course_code     = dc.course_code
    JOIN  dim_intake             di  ON r.intake_code     = di.intake_code
    LEFT JOIN dim_marketing_source dms ON ds.marketing_source_id = dms.marketing_source_id
""")
print(f"fact_result: {spark.table('fact_result').count()} rows")


## Verification

In [ ]:
gold_tables = [
    "dim_student", "dim_course", "dim_intake", "dim_marketing_source",
    "fact_lead", "fact_enrolment", "fact_result",
]

print(f"{'Table':<25} {'Rows':>6}")
print("-" * 33)
for tbl in gold_tables:
    print(f"{tbl:<25} {spark.table(tbl).count():>6}")

# Sanity: no NULL surrogate keys in facts
null_keys = spark.sql("""
    SELECT 'fact_lead'       AS fact, COUNT(*) AS null_student_keys FROM fact_lead       WHERE student_key IS NULL
    UNION ALL
    SELECT 'fact_enrolment', COUNT(*) FROM fact_enrolment WHERE student_key IS NULL OR course_key IS NULL OR intake_key IS NULL
    UNION ALL
    SELECT 'fact_result',    COUNT(*) FROM fact_result    WHERE student_key IS NULL OR course_key IS NULL OR intake_key IS NULL
""").collect()

print("\nNULL dimension key check (0 = pass):")
for row in null_keys:
    status = "PASS" if row[1] == 0 else "FAIL"
    print(f"  {row[0]:<20} {row[1]:>4} null keys  [{status}]")
